Data preprocessing


In [105]:
import pandas as pd
import numpy as np

# re is a regular expression library 
import re


Load unclean Dataset

In [106]:
df=pd.read_csv('../data/raw/lahore_properties.csv') 

Drop title Column

In [107]:
df = df.drop(columns=['Title'])

Price Cleaning


In [108]:
def parse_price_robust(price_str):
    if pd.isna(price_str):
        return price_str
        
    # Lowercase and remove ALL spaces to prevent mismatch errors
    clean_str = str(price_str).lower().replace(" ", "")
    
    match = re.search(r'([0-9\.]+)', clean_str)
    if not match:
        return None
        
    number = float(match.group(1))
    
    # Multiply based on keywords in the space-free string
    if 'lakh' in clean_str or 'lahk' in clean_str or 'lac' in clean_str:
        return number * 100000
    elif 'crore' in clean_str or 'cr' in clean_str:
        return number * 10000000
    elif 'arab' in clean_str:
        return number * 1000000000
    else:
        return number

# Create a new column to safely verify the conversion
df['Price'] = df['Price'].apply(parse_price_robust)

# View the original text next to the new computed values
print(df[['Price']].head(10)) 
    
    

         Price
0   81500000.0
1   15000000.0
2   24000000.0
3   75000000.0
4   69500000.0
5   15000000.0
6   49500000.0
7  140000000.0
8  165000000.0
9   20000000.0


Convert Bedrooms and Bathrooms in Float

In [109]:
# Example: Extracting numbers from Bedrooms and Bathrooms
df['Bedrooms'] = df['Bedrooms'].astype(str).str.extract(r'(\d+)').astype(float) 
df['Bathrooms'] = df['Bathrooms'].astype(str).str.extract(r'(\d+)').astype(float) 

df[['Bedrooms', 'Bathrooms', 'Price']].dtypes 

Bedrooms     float64
Bathrooms    float64
Price        float64
dtype: object

find the indexs of missing bedroom rows and bath room rows

In [110]:
# Find the row indexes where Bedrooms or Bathrooms are missing
missing_bedroom_indexes = df.index[df['Bedrooms'].isna()].tolist()
missing_bathroom_indexes = df.index[df['Bathrooms'].isna()].tolist()

print(f'Missing Bedrooms: {len(missing_bedroom_indexes)} rows')
print(missing_bedroom_indexes)
print(f'Missing Bathrooms: {len(missing_bathroom_indexes)} rows')
print(missing_bathroom_indexes)

Missing Bedrooms: 318 rows
[101, 108, 169, 354, 375, 529, 597, 598, 599, 933, 1101, 1342, 1444, 1600, 1799, 1894, 1923, 2078, 2084, 2130, 2149, 2236, 2239, 2257, 2586, 2593, 2683, 2689, 2719, 2720, 2722, 2743, 2744, 2746, 2914, 2958, 3135, 3136, 3190, 3191, 3208, 3223, 3226, 3332, 3558, 3682, 3707, 3875, 4099, 4100, 4142, 4229, 4246, 4294, 4297, 4576, 4651, 4658, 4732, 4842, 5070, 5210, 5256, 5322, 5405, 5423, 5499, 5844, 5845, 6125, 6270, 6271, 6272, 6521, 6553, 6558, 6659, 6698, 6700, 6731, 6745, 6757, 6768, 7000, 7014, 7089, 7090, 7095, 7163, 7172, 7211, 7218, 7239, 7263, 7264, 7449, 7501, 7521, 7522, 7523, 7524, 7525, 7526, 7792, 7815, 7885, 7892, 7911, 7999, 8026, 8148, 8177, 8396, 8405, 8498, 8506, 8528, 8549, 8686, 8990, 9214, 9469, 9626, 9631, 10006, 10101, 10160, 10263, 10269, 10349, 10479, 10492, 10677, 10873, 11176, 11223, 11385, 11396, 11398, 11643, 11669, 12077, 12099, 12204, 12309, 12311, 12312, 12313, 12314, 12316, 12317, 12321, 12323, 12324, 12336, 12338, 12339, 12340, 

Remove these rows

In [111]:
# Remove rows missing either Bedrooms or Bathrooms
rows_to_remove = df.index[df[['Bedrooms', 'Bathrooms']].isna().any(axis=1)]
df = df.drop(index=rows_to_remove).reset_index(drop=True)


print(f'Removed {len(rows_to_remove)} rows')
print(df[['Bedrooms', 'Bathrooms']].isna().sum())

Removed 441 rows
Bedrooms     0
Bathrooms    0
dtype: int64


In [91]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18249 entries, 0 to 18248
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Location             18249 non-null  str    
 1   Type                 18249 non-null  str    
 2   Price                18249 non-null  float64
 3   Area                 18249 non-null  str    
 4   Bedrooms             17931 non-null  float64
 5   Bathrooms            17829 non-null  float64
 6   Purpose              18249 non-null  str    
 7   Built Year           18249 non-null  float64
 8   Kitchens             18249 non-null  int64  
 9   Store Rooms          18249 non-null  int64  
 10  Servant Quarters     18249 non-null  int64  
 11  Furnished            18249 non-null  bool   
 12  Gym                  18249 non-null  bool   
 13  Study Room           18249 non-null  bool   
 14  Drawing Room         18249 non-null  bool   
 15  Dining Room          18249 non-null  bool   
 1

In [112]:
# Drop 'Date Posted'
df = df.drop(columns=['Date Posted']) 

# Impute 'Built Year' with the median
median_year = df['Built Year'].median() 
df['Built Year'] = df['Built Year'].fillna(median_year) 

In [104]:
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 17808 entries, 0 to 17807
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Location             17808 non-null  str    
 1   Type                 17808 non-null  str    
 2   Price                17808 non-null  float64
 3   Area                 17808 non-null  str    
 4   Bedrooms             17808 non-null  float64
 5   Bathrooms            17808 non-null  float64
 6   Purpose              17808 non-null  str    
 7   Built Year           17808 non-null  float64
 8   Kitchens             17808 non-null  int64  
 9   Store Rooms          17808 non-null  int64  
 10  Servant Quarters     17808 non-null  int64  
 11  Furnished            17808 non-null  bool   
 12  Gym                  17808 non-null  bool   
 13  Study Room           17808 non-null  bool   
 14  Drawing Room         17808 non-null  bool   
 15  Dining Room          17808 non-null  bool   
 1

In [113]:
# Convert binary columns to numeric values: TRUE -> 1 and FALSE -> 0
binary_columns = ['Furnished', 'Gym', 'Study Room', 'Drawing Room', 'Dining Room', 'Lawn/Garden', 'Swimming Pool', 'Electricity Backup', 'Lounge/Sitting Room']

for col in binary_columns:
    normalized_values = df[col].astype('string').str.strip().str.upper()
    df[col] = normalized_values.map({'TRUE': 1, 'FALSE': 0})

# Verify the conversion and save the processed data
print(df[binary_columns].stack().dropna().unique())
# df.to_excel(r'C:\Users\USER\Desktop\properties_price_ai\data\processed\clean.xlsx', index=False)

[1 0]


In [115]:
# print the gym column to verify the conversion
print(df['Gym'])

0        1
1        0
2        1
3        1
4        1
        ..
17803    0
17804    1
17805    1
17806    0
17807    0
Name: Gym, Length: 17808, dtype: int64


In [116]:
df.isnull().sum()

Location               0
Type                   0
Price                  0
Area                   0
Bedrooms               0
Bathrooms              0
Purpose                0
Built Year             0
Kitchens               0
Store Rooms            0
Servant Quarters       0
Furnished              0
Gym                    0
Study Room             0
Drawing Room           0
Dining Room            0
Lawn/Garden            0
Swimming Pool          0
Electricity Backup     0
Lounge/Sitting Room    0
dtype: int64

In [117]:
df.to_excel(r'C:\Users\USER\Desktop\properties_price_ai\data\processed\clean.xlsx', index=False) 